# Sepsis v2: повторы, измерения и признаки

Изучим, как группировка событий и повторы меняют сеть, насколько часто состояния повторяются,
какие признаки доступны при входе и улучшают ли они NLL на новых объектах.
В v2 независимые экспоненциальные времена по рёбрам конкурируют по минимуму;
признаки меняют **логарифмы интенсивностей**. Это описательный эксперимент по журналу, не клиническая модель.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "dima/survpm/src/survpm").is_dir())
sys.path.insert(0, str(ROOT / "dima/survpm/src"))
from survpm import (EventLog, Network, ProcessModel, ObjectStream, Simulator,
                    ExponentialTransitionModel, Forecaster, discover,
                    DirectlyFollowsBackend, evaluate, compare, compare_networks)
from survpm import features
pd.set_option("display.precision", 3, "display.max_columns", 20)

from survpm import datasets
PATH = ROOT / "Med_Eremina/Sepsis Cases - Event Log_grouped.csv"
START, TERMINAL = datasets.SEPSIS_START, datasets.SEPSIS_TERMINAL
RULES = {"starts": [START], "terminals": [TERMINAL], "repeats": "simultaneous"}

## 1. Полные трассы и прозрачные анализы

Используем существующую группировку `event_group`: Registration, Admission, Treatment,
Examination, Discharge. По умолчанию Examination остаётся состоянием, а значения анализов
также записаны в `measurements`. Прозрачный вариант удаляет Examination только из состояний.
При `simultaneous` склеиваются лишь одинаковые состояния в один момент;
при `collapse` склеивается вся последовательная серия одного состояния.

In [2]:
log = datasets.load_sepsis_log(PATH).prepare(RULES)
transparent = datasets.load_sepsis_log(
    PATH, measurement_activities=datasets.SEPSIS_MEASUREMENTS).prepare(RULES)
# Loader считает Age статическим и может взять первое значение из будущего.
# Для доступности на входе используем только первую строку Registration.
raw = pd.read_csv(PATH, dtype=str, keep_default_na=False)
registration = (raw.loc[raw.event_group == START]
                .sort_values("time:timestamp", kind="stable")
                .groupby("case_id", sort=False).head(1).set_index("case_id"))
registration_age = pd.to_numeric(registration.Age.replace("", np.nan), errors="coerce")
print("Age впервые позже регистрации (не используем на входе):",
      int((log.cases.Age.notna() & registration_age.reindex(log.cases.index).isna()).sum()))
for part in [log, transparent]:
    part.cases["Age"] = registration_age.reindex(part.cases.index)
collapsed = log.prepare(dict(RULES, repeats="collapse"))
print("Объектов:", len(log.cases), "событий:", len(log.events), "замеров:", len(log.measurements))
print("Статические:", log.static)
print("Динамические:", log.dynamic)
display(log.events.activity.value_counts().rename("events"))
display(log.measurements.groupby("feature")["value"].describe())

Age впервые позже регистрации (не используем на входе): 42
Объектов: 1050 событий: 8325 замеров: 7938
Статические: ['InfectionSuspected', 'DiagnosticBlood', 'DisfuncOrg', 'SIRSCritTachypnea', 'Hypotensie', 'SIRSCritHeartRate', 'Infusion', 'DiagnosticArtAstrup', 'Age', 'DiagnosticIC', 'DiagnosticSputum', 'DiagnosticLiquor', 'DiagnosticOther', 'SIRSCriteria2OrMore', 'DiagnosticXthorax', 'SIRSCritTemperature', 'DiagnosticUrinaryCulture', 'SIRSCritLeucos', 'Oligurie', 'DiagnosticLacticAcid', 'Diagnose', 'Hypoxie', 'DiagnosticUrinarySediment', 'DiagnosticECG']
Динамические: ['CRP', 'LacticAcid', 'Leucocytes']


activity
Examination     3684
Treatment       1511
Admission       1298
Registration    1050
Discharge        782
Name: events, dtype: int64

,count,mean,std,min,25%,50%,75%,max
feature,,,,,,,,
CRP,3123.0,113.746,86.775,5.0,43.0,95.0,162.0,573.0
LacticAcid,1454.0,1.949,1.479,0.2,1.1,1.6,2.3,14.9
Leucocytes,3361.0,12.949,14.995,0.2,7.7,11.0,15.0,381.3


## 2. Одинаковый holdout по целым объектам

В исходном журнале 1050 объектов. Для практичного демонстрационного запуска `MAX_CASES=300`:
сначала выбираем целые объекты фиксированным seed, затем делим их на 80% train и 20% test.
`MAX_CASES=None` включает всю выборку и увеличивает время выполнения.
Все варианты используют те же case_id.
Ни интервалы, ни замеры объекта не могут попасть в обе части. Конец наблюдения сохраняется.
Статические поля в исходном loader берутся из всей трассы. Даже Age иногда впервые появляется позже
регистрации: выше заменяем его только значением из **первой строки Registration**, а отсутствие
сохраняем как NaN. Будущим возрастом пропуски не заполняем. Infusion и диагностические флаги не используем:
они могут отражать последующее лечение или процесс регистрации данных.

In [3]:
def select_cases(log, ids):
    """Целые объекты: фильтруем все таблицы, включая конец наблюдения."""
    ids = set(ids)
    return EventLog(
        log.events.loc[log.events.case_id.isin(ids)].copy(),
        log.cases.loc[log.cases.index.isin(ids)].copy(),
        log.measurements.loc[log.measurements.case_id.isin(ids)].copy(),
        log.last_seen.loc[log.last_seen.index.isin(ids)].copy(),
        log.observation.loc[log.observation.case_id.isin(ids)].copy(),
        None if log.rules is None else dict(log.rules),
    )

MAX_CASES = 300  # None — все 1050 объектов; выбор до train/test и без знания исходов.
ids = np.random.default_rng(2026).permutation(log.cases.index.to_numpy())
if MAX_CASES is not None:
    ids = ids[:MAX_CASES]
print("Полная выборка:", len(log.cases), "демонстрационная:", len(ids))
boundary = int(.8 * len(ids))
train_ids, test_ids = ids[:boundary], ids[boundary:]
assert set(train_ids).isdisjoint(test_ids)
variants = {"simultaneous": log, "collapse": collapsed, "анализы прозрачны": transparent}
splits = {name: (select_cases(part, train_ids), select_cases(part, test_ids))
          for name, part in variants.items()}
train, test = splits["simultaneous"]
print("Train:", len(train.cases), "Test:", len(test.cases))

Полная выборка: 1050 демонстрационная: 300
Train: 240 Test: 60


## 3. Сеть и влияние подготовки

Для обучения задаём заранее известные пять групп состояний: все рёбра из нетерминальных
состояний в известные группы допустимы, Discharge поглощает. Для прозрачных анализов Examination
исключается. Это структурная оболочка, не утверждение, что все переходы встречаются в данных.
Интенсивность не наблюдавшегося на train ребра остаётся нулевой; новый такой переход в test даст
бесконечный NLL. Не используем тестовые рёбра для построения сети или сглаживания.

DirectlyFollowsBackend отдельно показывает наблюдаемую train-топологию без PM4Py.
S(t) сравнивается из Registration по 200 путям; это случайная оценка, и варианты подготовки задают
разные процессы. Их NLL нельзя ранжировать друг против друга из-за разных интервалов.

In [4]:
STATES = ["Registration", "Admission", "Treatment", "Examination", "Discharge"]

def known_network(transparent=False):
    states = [s for s in STATES if not (transparent and s == "Examination")]
    return Network.from_edges([(s, d) for s in states if s != TERMINAL for d in states],
                              starts=[START], terminals=[TERMINAL])

networks, baselines, rows = {}, {}, []
for name, (tr, te) in splits.items():
    network = known_network(transparent=name == "анализы прозрачны")
    fitted = ProcessModel(network).fit(tr)
    networks[name], baselines[name] = network, fitted
    found = discover(tr, DirectlyFollowsBackend(starts=[START], terminals=[TERMINAL]))
    iv = tr.intervals()
    cid = tr.events.case_id.iloc[0]
    as_of = tr.events.loc[tr.events.case_id == cid, "timestamp"].min()
    survival = Forecaster(fitted, n_paths=200, seed=42).survival(tr, as_of, [1., 7., 30.], case_id=cid)
    rows.append(dict(variant=name, intervals=len(iv), observed_edges=len(found.edges),
                     self_loops=int((found.edges.src == found.edges.dst).sum()),
                     **{f"S({int(t)})": s for t, s in zip(survival.horizon, survival.survival)}))
display(pd.DataFrame(rows).set_index("variant"))
model = baselines["simultaneous"]
display(model.edges_table().query("n_completed > 0"))

,intervals,observed_edges,self_loops,S(1),S(7),S(30)
variant,,,,,,
simultaneous,1685,14,3,0.89,0.33,0.015
collapse,1162,11,0,0.86,0.33,0.040
анализы прозрачны,883,8,2,0.87,0.37,0.015


,src,dst,n_completed,n_visits,rate,p,mean_days
2,Registration,Treatment,47,240,11.738,0.205,0.017
3,Registration,Examination,182,240,45.454,0.795,0.017
6,Admission,Admission,46,297,0.116,0.156,1.340
7,Admission,Treatment,4,297,0.010,0.014,1.340
8,Admission,Examination,215,297,0.544,0.729,1.340
9,Admission,Discharge,30,297,0.076,0.102,1.340
11,Treatment,Admission,134,346,8.193,0.414,0.050
12,Treatment,Treatment,130,346,7.948,0.401,0.050
13,Treatment,Examination,58,346,3.546,0.179,0.050
14,Treatment,Discharge,2,346,0.122,0.006,0.050


## 4. Немедленный повтор и возврат

Для базовой модели без признаков вероятности переходов постоянны: `p = rate / sum(rate)`.
Матрица ожидаемых посещений `N = (I − Q)⁻¹` для нетерминальных состояний даёт:
вероятность вернуться в s после ухода `1 − 1/N[s,s]`, ожидаемое число посещений при старте в s — `N[s,s]`.
Немедленный повтор — диагональ Q. Это небольшая описательная проверка модели без памяти.

В журнале берём только **завершённые** train-трассы: для незавершённых неизвестен будущий возврат.
Такое условие меняет состав выборки, поэтому эмпирические доли не являются несмещённой оценкой
для всей популяции. Сравнение с моделью помогает увидеть ограничения, а не доказывает качество прогноза.

In [5]:
def model_repeats(fitted):
    states = [s for s in fitted.network.states if s not in fitted.network.terminals]
    Q = pd.DataFrame(0., index=states, columns=states)
    for row in fitted.edges_table().itertuples():
        if row.dst in states:
            Q.loc[row.src, row.dst] = row.p
    N = np.linalg.inv(np.eye(len(states)) - Q.to_numpy())
    return pd.DataFrame({"state": states, "p_immediate_model": np.diag(Q),
                         "p_return_model": 1 - 1 / np.diag(N), "expected_visits": np.diag(N)})

def empirical_repeats(prepared):
    done = prepared.observation.loc[prepared.observation.event == 1, "case_id"]
    rows = []
    for cid, group in prepared.events.loc[prepared.events.case_id.isin(done)].groupby("case_id", sort=False):
        seq = group.activity.tolist()
        for k, state in enumerate(seq[:-1]):
            rows.append(dict(state=state, immediate=seq[k+1] == state, later=state in seq[k+1:]))
    return pd.DataFrame(rows).groupby("state").agg(
        n=("immediate", "size"), p_immediate_log=("immediate", "mean"), p_return_log=("later", "mean"))

repeats = model_repeats(model).set_index("state").join(empirical_repeats(train))
display(repeats)
display(model_repeats(baselines["collapse"]).set_index("state"))

,p_immediate_model,p_return_model,expected_visits,n,p_immediate_log,p_return_log
state,,,,,,
Admission,0.156,0.617,2.612,292,0.158,0.408
Examination,0.450,0.777,4.486,739,0.468,0.765
Registration,0.000,0.000,1.000,174,0.000,0.000
Treatment,0.401,0.685,3.173,290,0.383,0.466


,p_immediate_model,p_return_model,expected_visits
state,,,
Admission,0.0,0.546,2.204
Examination,0.0,0.595,2.467
Registration,0.0,0.000,1.000
Treatment,0.0,0.474,1.900


## 5. Что было известно при входе

`at_entry` использует последний замер не позже входа. Замер ровно в момент входа считается
доступным в этом состоянии. Более поздние анализы не меняют уже начавшееся ожидание.
В прозрачном варианте анализ внутри текущего состояния будет использован только при следующем входе.
Возраст и значения лабораторных показателей не заменяем будущим максимумом или средним по всей трассе.
История посещений и прошедшее время рассчитываются только по префиксу трассы.

In [6]:
iv = train.intervals()
entry_X = features.at_entry(iv, train, window="1D")
DYNAMIC = ["CRP", "Leucocytes", "LacticAcid"]
coverage = entry_X[DYNAMIC].notna().groupby(iv.src).mean()
coverage.insert(0, "n", iv.groupby("src").size())
display(coverage)
display(iv[["case_id", "src", "enter"]].join(entry_X[["Age", "CRP", "CRP_age", "CRP_n_1D"]]).head(8))

,n,CRP,Leucocytes,LacticAcid
src,,,,
Admission,297,0.949,1.000,0.926
Examination,802,0.939,0.999,0.898
Registration,240,0.017,0.025,0.000
Treatment,346,0.757,0.809,0.763


,case_id,src,enter,Age,CRP,CRP_age,CRP_n_1D
0,AG,Registration,2014-05-10 09:03:26+00:00,NaN,NaN,NaN,0
1,AG,Examination,2014-05-10 09:24:00+00:00,NaN,22.0,0.000,1
2,AKA,Registration,2014-08-02 13:20:57+00:00,35.0,NaN,NaN,0
3,AS,Registration,2013-11-23 16:32:23+00:00,75.0,NaN,NaN,0
4,AS,Examination,2013-11-23 17:02:00+00:00,75.0,55.0,0.000,1
5,AS,Treatment,2013-11-23 18:07:09+00:00,75.0,55.0,0.045,1
6,AS,Treatment,2013-11-23 18:07:22+00:00,75.0,55.0,0.045,1
7,AS,Examination,2013-11-23 20:39:00+00:00,75.0,50.0,0.000,2


## 6. Персонализированные интенсивности и NLL на holdout

Три модели используют одну структуру и одни и те же интервалы: без признаков; Age;
Age + последние анализы + история. В последней также используем давность и число замеров,
чтобы доступность анализа не терялась. Пропуски заполняются медианами с индикаторами;
кодирование и стандартизация обучаются только на train внутри `fit`. L2=1 задан заранее.

Оценка учитывает X на входе **каждого тестового интервала**, события и цензурированную экспозицию.
NLL меньше — лучше; неизвестные переходы не отбрасываются. Это одна воспроизводимая проверка,
без подбора признаков и регуляризации по test. Фактическое наблюдение незавершённых Sepsis-трасс
часто заканчивается последней записью; дополнительная экспозиция после неё неизвестна.
Если основной NLL бесконечен из-за нового ребра, отдельно показываем условное сравнение на общих
**целых тестовых объектах**, все наблюдаемые переходы которых были в train. Выбор покрытого поднабора
делается для диагностики после оценки, без переобучения. Его NLL не заменяет результат полного holdout;
число исключённых объектов показываем явно.


In [7]:
from IPython.display import Markdown

network = networks["simultaneous"]
FEATURES = (["Age"] + DYNAMIC + [f"{f}_age" for f in DYNAMIC]
            + [f"{f}_n" for f in DYNAMIC] + ["history_visits", "history_elapsed_days"])
age_model = ProcessModel(network, ExponentialTransitionModel(l2=1.)).fit(train, features=["Age"])
feature_model = ProcessModel(network, ExponentialTransitionModel(l2=1.)).fit(train, features=FEATURES)
models = {"без признаков": model, "Age": age_model, "Age + анализы + история": feature_model}
quality = compare(models, test, metrics=["nll"])
display(Markdown(f"**Основная оценка: полный holdout, {len(test.cases)} объектов.**"))
display(quality.pivot(index="model", columns="metric", values="value"))
train_pairs = set(train.intervals().loc[lambda d: d.observed, ["src", "dst"]].itertuples(index=False, name=None))
test_observed = test.intervals().loc[lambda d: d.observed]
unseen = test_observed.loc[[pair not in train_pairs for pair in test_observed[["src", "dst"]].itertuples(index=False, name=None)]]
print("Наблюдаемых тестовых интервалов с ребром, отсутствующим в train:", len(unseen))
uncovered_ids = set(unseen.case_id)
covered_ids = [cid for cid in test.cases.index if cid not in uncovered_ids]
print("Общие покрытые test-объекты:", len(covered_ids), "из", len(test.cases),
      "; исключено целых объектов:", len(uncovered_ids))
covered_test = select_cases(test, covered_ids)
conditional_quality = compare(models, covered_test, metrics=["nll"])
print("Условная оценка только на покрытых объектах; основной NLL выше относится ко всему holdout:")
display(Markdown(f"**Условная оценка: {len(covered_ids)} покрытых объектов из {len(test.cases)}.**"))
display(conditional_quality.pivot(index="model", columns="metric", values="value"))


Наблюдаемых тестовых интервалов с ребром, отсутствующим в train: 1
Общие покрытые test-объекты: 59 из 60 ; исключено целых объектов: 1
Условная оценка только на покрытых объектах; основной NLL выше относится ко всему holdout:


**Основная оценка: полный holdout, 60 объектов.**

metric,nll_per_interval,nll_per_trace
model,,
Age,inf,inf
Age + анализы + история,inf,inf
без признаков,inf,inf


**Условная оценка: 59 покрытых объектов из 60.**

metric,nll_per_interval,nll_per_trace
model,,
Age,1.220,7.860
Age + анализы + история,0.448,2.886
без признаков,1.090,7.020


## 7. Log-rate анализ

Коэффициенты относятся к стандартизованным train-признакам, включая индикаторы пропусков.
Положительный коэффициент увеличивает интенсивность данного выхода, `exp(coef)` — множитель.
Вероятность повтора зависит от его интенсивности **относительно всех конкурирующих выходов**.
По одному коэффициенту нельзя заключить, что вероятность повтора обязательно увеличилась.

Таблица ниже описывает модель после обучения; большие коэффициенты не являются причинным эффектом.
Это log-rate модель v2: выводы старых отдельных логистических и lognormal AFT экспериментов
к ней не переносятся. Исторические эксперименты сохранены в архиве.

In [8]:
def rate_coefficients(fitted):
    transition = fitted.transitions
    return pd.DataFrame([
        {"src": src, "dst": dst, "feature": name,
         "log_rate_coef": float(beta), "rate_ratio": float(np.exp(beta))}
        for (src, dst), betas in transition.coefficients.items()
        for name, beta in zip(transition.encoder.names_, betas)
    ])

coef = rate_coefficients(feature_model)
for src, table in coef.groupby("src", sort=False):
    # Самые большие по модулю коэффициенты, без отбора по test.
    top = table.loc[table.log_rate_coef.abs().nlargest(6).index]
    print(src)
    display(top.reset_index(drop=True))

Registration
Admission
Treatment
Examination


,src,dst,feature,log_rate_coef,rate_ratio
0,Registration,Treatment,CRP__missing,0.307,1.359
1,Registration,Treatment,CRP_age__missing,0.307,1.359
2,Registration,Treatment,CRP_n__missing,0.307,1.359
3,Registration,Treatment,Age,0.233,1.262
4,Registration,Treatment,Leucocytes__missing,-0.158,0.854
5,Registration,Treatment,Leucocytes_age__missing,-0.158,0.854


,src,dst,feature,log_rate_coef,rate_ratio
0,Admission,Admission,history_visits,-0.814,0.443
1,Admission,Discharge,CRP,-0.469,0.625
2,Admission,Admission,LacticAcid_n,-0.434,0.648
3,Admission,Discharge,CRP_n,0.405,1.500
4,Admission,Admission,LacticAcid,0.402,1.495
5,Admission,Treatment,Age,-0.356,0.700


,src,dst,feature,log_rate_coef,rate_ratio
0,Treatment,Treatment,history_visits,-5.536,0.004
1,Treatment,Admission,history_visits,1.634,5.124
2,Treatment,Admission,CRP_age,-0.903,0.405
3,Treatment,Admission,Leucocytes_age,-0.810,0.445
4,Treatment,Treatment,CRP_age,-0.723,0.485
5,Treatment,Treatment,Leucocytes_age,-0.672,0.510


,src,dst,feature,log_rate_coef,rate_ratio
0,Examination,Treatment,history_visits,-2.905,0.055
1,Examination,Treatment,CRP_n,-2.549,0.078
2,Examination,Treatment,Leucocytes_n,-2.401,0.091
3,Examination,Treatment,LacticAcid_age,-2.069,0.126
4,Examination,Treatment,history_elapsed_days,-2.014,0.133
5,Examination,Admission,history_visits,-1.217,0.296


## 8. Прогноз выбранного объекта на момент регистрации

Для прогноза пути используем персонализированную **Age-модель**: возраст известен на регистрации
и сохраняется на всём будущем пути. Берём только первый вход тестового объекта.
Forecaster ограничивает реальные наблюдения `as_of`, поэтому поздние события и анализы недоступны.

Полная модель с лабораторными значениями и историей оценена выше по входам реальных holdout интервалов.
Её свободная симуляция требует отдельной проверки будущих сценариев анализов и диапазонов истории:
при экспериментальном запуске экстраполяция дала экстремальные интенсивности, и Simulator
отклонил шаг, не продвигающий время. Это ограничение не скрываем заменой значений или отсечением траекторий;
полную модель здесь не используем для прогноза пути.
200 путей дают случайную погрешность и не являются клинической валидацией.


In [9]:
case_id = test.events.case_id.iloc[0]
as_of = test.events.loc[test.events.case_id == case_id, "timestamp"].min()
ctx = age_model.history_features.transform(test, case_id, as_of)
print("Объект:", case_id, "as_of:", as_of, "состояние:", ctx["state"])
print("Age на входе:", ctx["X"]["Age"])
display(Forecaster(age_model, n_paths=200, seed=42).survival(
    test, as_of, [1., 7., 30.], case_id=case_id))


Объект: AAA as_of: 2014-11-19 02:16:21+00:00 состояние: Registration
Age на входе: 60.0


,horizon,survival,cdf
0,1.0,0.870,0.130
1,7.0,0.325,0.675
2,30.0,0.010,0.990


Результаты читаются из выполненных таблиц: подготовка меняет число интервалов и повторов;
модель без памяти задаёт простую точку сравнения; пользу признаков оцениваем на общих holdout объектах.
Если у модели с признаками NLL выше, этот набор и регуляризация не улучшили проверку.
Один split не даёт устойчивого медицинского вывода; для нового исследования нужны повторные
разбиения целых объектов и отдельная внешняя проверка. Этот ноутбук не подбирает модель по test.